In [ ]:
# ==============================================================================
# SYMBOLIC REGRESSION AND SMOTE
# ===============================================================================

In [2]:

# ==============================================================================
# IMPORTS, PATCHES, WRAPPER
# ==============================================================================


In [ ]:

import os
import gc
import sys
import json
import time
import warnings
from collections import Counter, defaultdict

import joblib
import numpy as np
import pandas as pd
import sklearn.utils
import gplearn.genetic

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("default")

# Print and flush messages immediately so symbolic-regression runs show progress in real time.

def log(msg):
    print(msg, flush=True)
    sys.stdout.flush()



In [ ]:
# Patch 1: sklearn.utils.check_random_state

if not hasattr(sklearn.utils, "check_random_state"):
    try:
        from sklearn.utils.validation import check_random_state
        sklearn.utils.check_random_state = check_random_state
        log("Patch 1/2: sklearn.utils.check_random_state... OK")
    except ImportError:
        log("Patch 1/2: Failed to expose sklearn.utils.check_random_state")



In [ ]:
# Patch 2: _validate_data compatibility
# Compatibility shim for sklearn validation APIs across different sklearn versions.

def _compat_validate_data(self, X, y=None, reset=True, **kwargs):
    try:
        from sklearn.utils.validation import validate_data as skl_validate_data
        if y is None:
            return skl_validate_data(self, X=X, reset=reset, **kwargs)
        return skl_validate_data(self, X=X, y=y, reset=reset, **kwargs)
    except Exception:
        from sklearn.utils.validation import check_X_y, check_array
        local_kwargs = dict(kwargs)
        local_kwargs.pop("reset", None)
        if y is None:
            X_checked = check_array(X, **local_kwargs)
            if reset or not hasattr(self, "n_features_in_"):
                self.n_features_in_ = X_checked.shape[1]
            return X_checked
        X_checked, y_checked = check_X_y(X, y, **local_kwargs)
        if reset or not hasattr(self, "n_features_in_"):
            self.n_features_in_ = X_checked.shape[1]
        return X_checked, y_checked



In [6]:

if not hasattr(gplearn.genetic.BaseSymbolic, "_validate_data"):
    gplearn.genetic.BaseSymbolic._validate_data = _compat_validate_data
    log("Patch 2/2: BaseSymbolic._validate_data... OK")
else:
    log("Patch 2/2: BaseSymbolic._validate_data already available... OK")



Patch 2/2: BaseSymbolic._validate_data... OK


In [ ]:
# Robust wrapper: makes sklearn scorers understand this as a classifier
# Wrapper that marks gplearn's symbolic classifier as a proper binary sklearn classifier.

class RobustSymbolicClassifier(gplearn.genetic.SymbolicClassifier):
    _estimator_type = "classifier"

    # Provide sklearn estimator tags, including binary-only behavior and required target labels.
    def _more_tags(self):
        tags = {}
        parent_more_tags = getattr(super(), "_more_tags", None)
        if callable(parent_more_tags):
            try:
                tags.update(parent_more_tags())
            except Exception:
                pass
        tags.update({"binary_only": True, "requires_y": True})
        return tags

    # Update newer sklearn tag objects so the symbolic model is recognized as a classifier.
    def __sklearn_tags__(self):
        parent_tags_method = getattr(super(), "__sklearn_tags__", None)
        if callable(parent_tags_method):
            tags = parent_tags_method()
            if hasattr(tags, "estimator_type"):
                tags.estimator_type = "classifier"
            if hasattr(tags, "target_tags") and hasattr(tags.target_tags, "required"):
                tags.target_tags.required = True
            if hasattr(tags, "classifier_tags") and tags.classifier_tags is not None:
                if hasattr(tags.classifier_tags, "multi_class"):
                    tags.classifier_tags.multi_class = False
            return tags
        return self._more_tags()


log("Custom Wrapper Created: RobustSymbolicClassifier... OK")



Custom Wrapper Created: RobustSymbolicClassifier... OK


In [8]:

# ==============================================================================
# CONFIG
# ==============================================================================


In [9]:

SEED = 42
RAW_DATA_PATH = os.path.join("..", "data", "processed", "dados_uteis_refreshed_v3_memorysafe.pkl")
OUTPUT_DIR = os.path.join("..", "data", "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COL = "ausencia"
RUN_TAG = "sr_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained"

DROP_COLS = [
    "cns_usuario", "ibge_usuario", "cep_usuario", "cep_cnes_exe",
    "ibge_cnes_exe", "lat_cnes", "lon_cnes", "lat_usuario", "lon_usuario",
    "cep_aprox_usuario", "cep_aprox_cnes", "dta_atend", "cod_uf_usuario",
    "cod_uf_cnes", "cidade_usuario", "cidade_cnes"
]

KNOWN_CATEGORICAL_COLS = [
    "sexo_usuario",
    "raca_cor_usuario",
    "cbo_espec",
    "prioridade",
    "deslocamento",
    "tip_marcacao",
    "uf_usuario",
    "uf_cnes",
]


In [ ]:

# ----------------------------------------------------------------------------
# Deina-style nested CV config
# ----------------------------------------------------------------------------
N_OUTER_FOLDS = 10  # z1 in Step 2
N_INNER_FOLDS = 10  # z2 in Step 4
TOP_K_FEATURES = 20


In [11]:

# Memory-safe defaults. Set to None to make the folds use the full available data.
# Symbolic Regression + nested pure SMOTE is very expensive, so these caps are intentional.
OUTER_CALIBRATION_CAP = 300_000
INNER_TRAIN_CAP = 80_000


In [12]:

# Final outer symbolic training uses train-side monitor/threshold subsets only.
FINAL_MONITOR_SIZE = 30_000
FINAL_THRESHOLD_SIZE = 30_000
FINAL_TRAIN_CAP = 150_000


In [13]:

# ----------------------------------------------------------------------------
# Pure SMOTE config
# ----------------------------------------------------------------------------
SMOTE_SAMPLING_STRATEGY = 0.50  # minority_after / majority = 0.5 -> about 1:2
SMOTE_K_NEIGHBORS = 3


In [14]:

# ----------------------------------------------------------------------------
# Feature ranking config: permutation importance replaces wrapper in Step 5
# ----------------------------------------------------------------------------
FEATURE_RANK_MODEL_TRAIN_CAP = 80_000
FEATURE_RANK_PERM_CAP = 20_000
FEATURE_RANK_PERM_REPEATS = 3


In [15]:

# ----------------------------------------------------------------------------
# Symbolic Regression training config
# ----------------------------------------------------------------------------
N_JOBS_FINAL = 2
MODEL_VERBOSE = 0
TOTAL_GENERATIONS = 35
GENERATION_STEP = 5
EARLY_STOPPING_PATIENCE = 4
MIN_AUC_IMPROVEMENT = 0.0005


In [16]:

BASE_FIXED_PARAMS = {
    "population_size": 1000,
    "tournament_size": 30,
    "stopping_criteria": 0.0,
    "const_range": (-5.0, 5.0),
    "init_depth": (2, 6),
    "init_method": "grow",
    "function_set": ["add", "sub", "mul", "div", "abs", "neg", "inv"],
    "transformer": "sigmoid",
    "metric": "log loss",
    "parsimony_coefficient": 0.0005,
    "p_crossover": 0.90,
    "p_subtree_mutation": 0.01,
    "p_hoist_mutation": 0.01,
    "p_point_mutation": 0.01,
    "p_point_replace": 0.05,
    "max_samples": 1.0,
    "class_weight": None,
    "warm_start": True,
    "low_memory": True,
    "n_jobs": N_JOBS_FINAL,
    "verbose": MODEL_VERBOSE,
    "random_state": SEED,
}


In [17]:

# ----------------------------------------------------------------------------
# Threshold config
# ----------------------------------------------------------------------------
THRESHOLD_OBJECTIVE = "youden_j"  # fixed strategy: Youden + calibration + constraints
MANUAL_THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
MIN_THRESHOLD = 0.10
MIN_SPECIFICITY_FOR_THRESHOLD = 0.20
MIN_PRECISION_MARGIN_OVER_PREVALENCE = 0.00
MAX_PREDICTED_POSITIVE_RATE = 0.70
MIN_PREDICTED_NEGATIVE_RATE = 0.05
USE_SIGMOID_PROBA_CALIBRATION = True


In [18]:

# ----------------------------------------------------------------------------
# Optional final best-fold permutation importance on selected features
# ----------------------------------------------------------------------------
RUN_OUTER_BEST_FOLD_PERMUTATION = False
OUTER_PERM_MAX_SAMPLES = 20_000
OUTER_PERM_REPEATS = 3


In [19]:


# ==============================================================================
# HELPERS
# ==============================================================================


In [ ]:
# Build a dense OneHotEncoder while supporting both old and new sklearn parameter names.

def make_dense_ohe(categories=None):
    kwargs = {"handle_unknown": "ignore"}
    if categories is not None:
        kwargs["categories"] = categories
    try:
        return OneHotEncoder(sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=False, **kwargs)



In [ ]:
# Convert arrays to float32 to reduce memory use during symbolic regression and SMOTE.

def to_float32_array(X):
    return np.asarray(X, dtype=np.float32)


In [ ]:
# Coerce labels safely to integer 0/1 and fail early if labels are missing or not binary.

def ensure_binary_int_labels(y, name="y"):
    if isinstance(y, pd.Series):
        arr = y.to_numpy()
    else:
        arr = np.asarray(y)
    arr = arr.ravel()
    if pd.isna(arr).any():
        raise ValueError(f"{name} contains missing values.")
    try:
        arr_num = pd.to_numeric(pd.Series(arr), errors="raise").to_numpy()
    except Exception:
        uniques = pd.Series(arr).dropna().unique().tolist()
        if len(uniques) != 2:
            raise ValueError(f"{name} is not binary and could not be coerced safely.")
        mapping = {uniques[0]: 0, uniques[1]: 1}
        arr_num = pd.Series(arr).map(mapping).to_numpy()
    arr_int = arr_num.astype(np.int64)
    unique_vals = np.unique(arr_int)
    valid_sets = [np.array([0, 1]), np.array([0]), np.array([1])]
    if not any(np.array_equal(unique_vals, s) for s in valid_sets):
        raise ValueError(f"{name} must contain only 0/1 after coercion. Found: {unique_vals}")
    return arr_int



In [ ]:
# Return class-0 and class-1 counts after enforcing binary labels.

def get_class_counts(y):
    y = ensure_binary_int_labels(y, "y_counts")
    counts = np.bincount(y, minlength=2)
    return int(counts[0]), int(counts[1])



In [ ]:
# Log total samples, class counts, percentages, and imbalance ratio.

def print_class_distribution(name, y):
    c0, c1 = get_class_counts(y)
    total = c0 + c1
    p0 = c0 / total * 100 if total else 0.0
    p1 = c1 / total * 100 if total else 0.0
    ratio = np.inf if min(c0, c1) == 0 else max(c0, c1) / min(c0, c1)
    log(f"\n{name}")
    log("-" * len(name))
    log(f"Total: {total:,}".replace(",", "."))
    log(f"Class 0: {c0:,} ({p0:.4f}%)".replace(",", "."))
    log(f"Class 1: {c1:,} ({p1:.4f}%)".replace(",", "."))
    if np.isinf(ratio):
        log("Imbalance ratio: infinito")
    else:
        log(f"Imbalance ratio (maj/min): {ratio:.4f}:1")



In [ ]:
# Return TN, FP, FN, TP with fixed 0/1 labels, even when one class is absent.

def safe_binary_confusion(y_true, y_pred):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_pred = ensure_binary_int_labels(y_pred, "y_pred")
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape != (2, 2):
        padded = np.zeros((2, 2), dtype=int)
        padded[:cm.shape[0], :cm.shape[1]] = cm
        cm = padded
    tn, fp, fn, tp = cm.ravel()
    return tn, fp, fn, tp



In [ ]:
# Compute specificity, TN / (TN + FP), with a safe zero-denominator fallback.

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)
    return 0.0 if (tn + fp) == 0 else tn / (tn + fp)



In [ ]:
# Compute negative predictive value and return NaN when no predicted negatives exist.

def npv_score(y_true, y_pred):
    """Negative Predictive Value: TN / (TN + FN).

    If there are no predicted negatives, NPV is mathematically undefined.
    Returning np.nan is more honest than returning 0.0, because 0.0 would mean
    the model made negative predictions and all of them were wrong.
    """
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)
    denom = tn + fn
    return np.nan if denom == 0 else tn / denom



In [ ]:
# Compute ROC AUC, returning NaN instead of crashing when AUC is undefined.

def safe_roc_auc(y_true, y_proba):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    try:
        return roc_auc_score(y_true, y_proba)
    except ValueError:
        return np.nan



In [ ]:
# Collect fold metrics, confusion counts, class rates, and degenerate-prediction flags.

def calculate_metrics(y_true, y_pred, y_proba):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_pred = ensure_binary_int_labels(y_pred, "y_pred")
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)

    real_neg = tn + fp
    real_pos = tp + fn
    pred_neg = tn + fn
    pred_pos = tp + fp
    total = real_neg + real_pos

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": safe_roc_auc(y_true, y_proba),

        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),

        "Real_Positive": int(real_pos),
        "Real_Negative": int(real_neg),
        "Predicted_Positive": int(pred_pos),
        "Predicted_Negative": int(pred_neg),

        "Real_Positive_Rate": real_pos / total if total else np.nan,
        "Real_Negative_Rate": real_neg / total if total else np.nan,
        "Predicted_Positive_Rate": pred_pos / total if total else np.nan,
        "Predicted_Negative_Rate": pred_neg / total if total else np.nan,

        "Has_No_Predicted_Negatives": bool(pred_neg == 0),
        "Has_No_Predicted_Positives": bool(pred_pos == 0),
    }



In [ ]:
# Check that manually supplied thresholds are inside (0, 1) and remove duplicates.

def validate_manual_thresholds(thresholds):
    valid = []
    for t in thresholds:
        t = float(t)
        if not (0.0 < t < 1.0):
            raise ValueError(f"Threshold {t} is invalid. Thresholds must be between 0 and 1.")
        valid.append(t)
    return sorted(set(valid))



In [ ]:
# Select the best threshold while blocking operationally useless all-positive or all-negative behavior.

def optimize_threshold_constrained(
    y_true,
    y_proba,
    objective="youden_j",
    min_threshold=MIN_THRESHOLD,
    min_specificity=MIN_SPECIFICITY_FOR_THRESHOLD,
    min_precision=None,
    precision_margin_over_prevalence=MIN_PRECISION_MARGIN_OVER_PREVALENCE,
    max_predicted_positive_rate=MAX_PREDICTED_POSITIVE_RATE,
    min_predicted_negative_rate=MIN_PREDICTED_NEGATIVE_RATE,
):
    """Choose a threshold while blocking pathological all-positive/all-negative behavior.

    Youden's J balances recall and specificity. The constraints are intentionally
    mild and only prevent thresholds that make the classifier operationally useless.
    """
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_proba = np.asarray(y_proba, dtype=np.float64)

    prevalence = float(np.mean(y_true))
    if min_precision is None:
        min_precision = prevalence + precision_margin_over_prevalence

    grid = np.linspace(min_threshold, 0.99, 90)
    quantiles = np.quantile(y_proba, np.linspace(0.01, 0.99, 99))
    manual = validate_manual_thresholds(MANUAL_THRESHOLDS)
    thresholds = np.unique(np.concatenate([grid, quantiles, manual]))
    thresholds = thresholds[(thresholds >= min_threshold) & (thresholds < 1.0)]

    rows = []
    for thr in thresholds:
        y_pred = (y_proba >= thr).astype(np.uint8)
        tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)

        total = len(y_true)
        pred_pos = tp + fp
        pred_neg = tn + fn
        pred_pos_rate = pred_pos / total if total else np.nan
        pred_neg_rate = pred_neg / total if total else np.nan

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        specificity = specificity_score(y_true, y_pred)
        npv = npv_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        youden_j = recall + specificity - 1.0
        balanced_accuracy = (recall + specificity) / 2.0

        passes_constraints = (
            specificity >= min_specificity
            and precision >= min_precision
            and pred_pos_rate <= max_predicted_positive_rate
            and pred_neg_rate >= min_predicted_negative_rate
        )

        rows.append({
            "Threshold": float(thr),
            "Precision": float(precision),
            "Recall": float(recall),
            "Specificity": float(specificity),
            "NPV": float(npv) if not pd.isna(npv) else np.nan,
            "F1": float(f1),
            "Youden_J": float(youden_j),
            "Balanced_Accuracy": float(balanced_accuracy),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp),
            "Predicted_Positive": int(pred_pos),
            "Predicted_Negative": int(pred_neg),
            "Predicted_Positive_Rate": float(pred_pos_rate),
            "Predicted_Negative_Rate": float(pred_neg_rate),
            "Min_Precision_Required": float(min_precision),
            "Min_Specificity_Required": float(min_specificity),
            "Max_Predicted_Positive_Rate_Allowed": float(max_predicted_positive_rate),
            "Min_Predicted_Negative_Rate_Required": float(min_predicted_negative_rate),
            "Passes_Constraints": bool(passes_constraints),
        })

    df_thr = pd.DataFrame(rows)
    if objective.lower() == "f1":
        score_col = "F1"
    elif objective.lower() == "youden_j":
        score_col = "Youden_J"
    else:
        raise ValueError("objective must be 'f1' or 'youden_j'")

    df_valid = df_thr[df_thr["Passes_Constraints"]].copy()
    threshold_selection_mode = "constrained"

    if len(df_valid) == 0:
        df_valid = df_thr[(df_thr["Predicted_Negative"] > 0) & (df_thr["Specificity"] > 0)].copy()
        threshold_selection_mode = "fallback_non_degenerate"

    if len(df_valid) == 0:
        df_valid = df_thr.copy()
        threshold_selection_mode = "fallback_all_thresholds"

    best_row = (
        df_valid.sort_values(
            by=[score_col, "Specificity", "Precision", "Recall", "Threshold"],
            ascending=[False, False, False, False, True],
        )
        .iloc[0]
        .to_dict()
    )
    best_row["Threshold_Selection_Mode"] = threshold_selection_mode
    best_row["Threshold_Valid_Candidates"] = int(len(df_valid))
    best_row["Threshold_Total_Candidates"] = int(len(df_thr))
    return best_row, df_thr



In [ ]:
# Clip probabilities away from 0 and 1 so logit calibration stays numerically stable.

def clip_proba(p, eps=1e-6):
    return np.clip(np.asarray(p, dtype=np.float64), eps, 1.0 - eps)



In [ ]:
# Convert probabilities to log-odds for sigmoid/Platt-style calibration.

def logit(p):
    p = clip_proba(p)
    return np.log(p / (1.0 - p))



In [ ]:
# Fit a one-dimensional calibrator on natural-distribution data, not SMOTEd data.

def fit_sigmoid_calibrator(y_calib, proba_calib, seed=SEED):
    """Fit a one-dimensional sigmoid/Platt-style calibrator on natural-distribution data."""
    y_calib = ensure_binary_int_labels(y_calib, "y_calib")
    if len(np.unique(y_calib)) < 2:
        return None
    z = logit(proba_calib).reshape(-1, 1)
    calibrator = LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=seed,
    )
    calibrator.fit(z, y_calib)
    return calibrator



In [ ]:
# Apply the fitted probability calibrator, or return raw probabilities if calibration was skipped.

def apply_sigmoid_calibrator(calibrator, proba):
    if calibrator is None:
        return np.asarray(proba, dtype=np.float64)
    z = logit(proba).reshape(-1, 1)
    return calibrator.predict_proba(z)[:, 1]



In [ ]:
# Pool TP, TN, FP, and FN across folds and recompute global confusion-derived metrics.

def summarize_pooled_confusion(df_outer):
    required = ["TP", "TN", "FP", "FN"]
    if not all(c in df_outer.columns for c in required):
        return pd.DataFrame()

    total_tp = int(df_outer["TP"].sum())
    total_tn = int(df_outer["TN"].sum())
    total_fp = int(df_outer["FP"].sum())
    total_fn = int(df_outer["FN"].sum())
    total = total_tp + total_tn + total_fp + total_fn

    accuracy = (total_tp + total_tn) / total if total else np.nan
    precision = np.nan if (total_tp + total_fp) == 0 else total_tp / (total_tp + total_fp)
    recall = np.nan if (total_tp + total_fn) == 0 else total_tp / (total_tp + total_fn)
    specificity = np.nan if (total_tn + total_fp) == 0 else total_tn / (total_tn + total_fp)
    npv = np.nan if (total_tn + total_fn) == 0 else total_tn / (total_tn + total_fn)
    f1 = np.nan if pd.isna(precision) or pd.isna(recall) or (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    balanced_accuracy = np.nan if pd.isna(recall) or pd.isna(specificity) else (recall + specificity) / 2.0

    return pd.DataFrame([{
        "TP": total_tp,
        "TN": total_tn,
        "FP": total_fp,
        "FN": total_fn,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "NPV": npv,
        "F1": f1,
        "Balanced_Accuracy": balanced_accuracy,
        "Predicted_Positive": total_tp + total_fp,
        "Predicted_Negative": total_tn + total_fn,
        "Real_Positive": total_tp + total_fn,
        "Real_Negative": total_tn + total_fp,
        "Predicted_Positive_Rate": (total_tp + total_fp) / total if total else np.nan,
        "Predicted_Negative_Rate": (total_tn + total_fn) / total if total else np.nan,
    }])



In [ ]:
# Subset labels correctly whether y is a pandas object or a NumPy array.

def subset_y(y, indices):
    return y.iloc[indices] if hasattr(y, "iloc") else y[indices]



In [ ]:
# Create a stratified capped subset to keep large operations memory-safe.

def make_cap_subset_indices(y, cap, seed):
    y = ensure_binary_int_labels(y, "y_cap")
    if cap is None or len(y) <= cap:
        return np.arange(len(y))
    indices = np.arange(len(y))
    _, idx_small = train_test_split(
        indices,
        test_size=cap,
        stratify=y,
        random_state=seed,
    )
    return idx_small



In [ ]:
# Identify categorical columns using known column names plus object/string/bool/category dtypes.

def infer_categorical_columns(X_df):
    known = {c for c in KNOWN_CATEGORICAL_COLS if c in X_df.columns}
    detected = set()
    for col in X_df.columns:
        s = X_df[col]
        if (
            pd.api.types.is_object_dtype(s)
            or pd.api.types.is_string_dtype(s)
            or pd.api.types.is_bool_dtype(s)
            or isinstance(s.dtype, pd.CategoricalDtype)
        ):
            detected.add(col)
    return [col for col in X_df.columns if col in (known | detected)]



In [ ]:
# Normalize raw features into numeric float32 columns and categorical columns with fixed categories.

def clean_raw_like_dataframe(X_df, num_feats, cat_feats, cat_categories=None):
    X_out = pd.DataFrame(index=X_df.index)
    for col in num_feats:
        X_out[col] = pd.to_numeric(X_df[col], errors="coerce").astype(np.float32)
    for i, col in enumerate(cat_feats):
        s = X_df[col]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype("object")
        s = s.where(~pd.isna(s), "__MISSING__").astype(str)
        if cat_categories is not None:
            cats = cat_categories[i]
            X_out[col] = pd.Categorical(s, categories=cats)
        else:
            X_out[col] = s.astype("category")
    return X_out



In [ ]:
# Learn numeric and categorical schema, adding missing and unknown category placeholders.

def build_outer_schema(X_outer_raw):
    cat_feats = infer_categorical_columns(X_outer_raw)
    num_feats = [c for c in X_outer_raw.columns if c not in cat_feats]
    cat_categories = []
    for col in cat_feats:
        s = X_outer_raw[col]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype("object")
        s = s.where(~pd.isna(s), "__MISSING__").astype(str)
        cats = sorted(pd.unique(s).tolist())
        if "__UNKNOWN__" not in cats:
            cats.append("__UNKNOWN__")
        cat_categories.append(cats)
    return num_feats, cat_feats, cat_categories



In [ ]:
# Build MinMax scaling plus fixed-category one-hot preprocessing to avoid leakage.

def build_preprocessor_from_schema(num_feats, cat_feats, cat_categories):
    return ColumnTransformer(
        transformers=[
            ("num", MinMaxScaler(), num_feats),
            ("cat", make_dense_ohe(categories=cat_categories), cat_feats),
        ],
        verbose_feature_names_out=False,
    )



In [ ]:
# Create forward and inverse integer maps for categorical values.

def learn_category_maps_from_schema(cat_feats, cat_categories):
    maps = {}
    inverse_maps = {}
    for col, cats in zip(cat_feats, cat_categories):
        mapping = {str(cat): i for i, cat in enumerate(cats)}
        inverse = {i: str(cat) for cat, i in mapping.items()}
        maps[col] = mapping
        inverse_maps[col] = inverse
    return maps, inverse_maps



In [ ]:
# Convert raw categorical values to integer codes while sending unseen values to UNKNOWN.

def apply_category_maps(X_df, cat_feats, num_feats, category_maps):
    X_out = pd.DataFrame(index=X_df.index)
    for col in num_feats:
        X_out[col] = pd.to_numeric(X_df[col], errors="coerce").astype(np.float32)
    for col in cat_feats:
        s = X_df[col]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype("object")
        s = s.where(~pd.isna(s), "__MISSING__").astype(str)
        X_out[col] = s.map(category_maps[col]).fillna(category_maps[col].get("__UNKNOWN__", -1)).astype(np.int32)
    return X_out



In [ ]:
# Build an ordered numeric matrix and categorical index list for SMOTE-related processing.

def build_smote_matrix(df_mapped, num_feats, cat_feats):
    ordered_cols = list(num_feats) + list(cat_feats)
    X_num = df_mapped[ordered_cols].copy()
    for col in num_feats:
        X_num[col] = pd.to_numeric(X_num[col], errors="coerce").astype(np.float32)
    for col in cat_feats:
        X_num[col] = pd.to_numeric(X_num[col], errors="coerce").fillna(-1).astype(np.int32)
    X_matrix = X_num.to_numpy(dtype=np.float32, copy=True)
    categorical_indices = [ordered_cols.index(col) for col in cat_feats]
    return X_matrix, ordered_cols, categorical_indices



In [ ]:
# Convert resampled numeric output back to a DataFrame and round categorical codes.

def restore_smote_dataframe(X_res, ordered_cols, num_feats, cat_feats):
    X_res_df = pd.DataFrame(X_res, columns=ordered_cols)
    for col in num_feats:
        X_res_df[col] = pd.to_numeric(X_res_df[col], errors="coerce").astype(np.float32)
    for col in cat_feats:
        X_res_df[col] = (
            pd.to_numeric(X_res_df[col], errors="coerce")
            .fillna(-1)
            .round()
            .astype(np.int32)
        )
    return X_res_df



In [ ]:
# Convert mapped categorical codes back to raw-like categorical columns.

def inverse_category_maps_to_raw_like(X_df_mapped, cat_feats, num_feats, inverse_maps, cat_categories):
    X_out = pd.DataFrame(index=X_df_mapped.index)
    for col in num_feats:
        X_out[col] = pd.to_numeric(X_df_mapped[col], errors="coerce").astype(np.float32)
    cat_to_categories = {col: cats for col, cats in zip(cat_feats, cat_categories)}
    for col in cat_feats:
        values = pd.to_numeric(X_df_mapped[col], errors="coerce").fillna(-1).round().astype(np.int32)
        restored = values.map(inverse_maps[col]).fillna("__UNKNOWN__").astype(str)
        X_out[col] = pd.Categorical(restored, categories=cat_to_categories[col])
    return X_out



In [ ]:
# Apply pure SMOTE on the numeric preprocessed matrix; fractional one-hot values are acceptable for symbolic numeric modeling.

def apply_pure_smote_array(X_train, y_train, seed, label="SMOTE"):
    """Apply pure SMOTE to the fully numeric MinMax + one-hot matrix.

    This intentionally replaces Pure SMOTE because the feature matrix is already
    numeric after preprocessing. Important: synthetic one-hot columns may become
    fractional, which is acceptable for Symbolic Regression as a continuous model
    but should be interpreted as synthetic numeric representation rather than a
    literal raw categorical record.
    """
    y_train = ensure_binary_int_labels(y_train, f"y_train_for_{label}")
    y_before = y_train.copy()

    smote = SMOTE(
        sampling_strategy=SMOTE_SAMPLING_STRATEGY,
        random_state=seed,
        k_neighbors=SMOTE_K_NEIGHBORS,
    )

    start = time.time()
    X_res, y_res = smote.fit_resample(X_train, y_train)
    elapsed_min = (time.time() - start) / 60.0
    X_res = to_float32_array(X_res)
    y_res = ensure_binary_int_labels(y_res, f"y_resampled_{label}")

    c0_before, c1_before = get_class_counts(y_before)
    c0_after, c1_after = get_class_counts(y_res)
    log(
        f"{label} finished in {elapsed_min:.2f}m | "
        f"before class0={c0_before}, class1={c1_before} | "
        f"after class0={c0_after}, class1={c1_after}"
    )

    return X_res, y_res, smote, elapsed_min



In [ ]:
# Create a stratified capped training subset for the feature-ranking baseline model.

def make_feature_rank_train_subset(X, y, max_size, seed):
    y = ensure_binary_int_labels(y, "y_rank_train")
    if len(X) <= max_size:
        return X, y
    idx = np.arange(len(X))
    _, idx_small = train_test_split(idx, test_size=max_size, stratify=y, random_state=seed)
    return X[idx_small], y[idx_small]



In [ ]:
# Create a stratified capped evaluation subset for feature-ranking permutation importance.

def make_feature_rank_eval_subset(X, y, max_size, seed):
    y = ensure_binary_int_labels(y, "y_rank_eval")
    if len(X) <= max_size:
        return X, y
    idx = np.arange(len(X))
    _, idx_small = train_test_split(idx, test_size=max_size, stratify=y, random_state=seed)
    return X[idx_small], y[idx_small]



In [ ]:
# Scorer used by permutation importance, based on predicted class-1 probabilities.

def roc_auc_proba_scorer(estimator, X, y):
    y = ensure_binary_int_labels(y, "y_score")
    y_proba = estimator.predict_proba(X)[:, 1]
    return roc_auc_score(y, y_proba)



In [ ]:
# Train a lightweight logistic baseline and compute AUC-based permutation importances for feature ranking.

def compute_lr_permutation_importance_vector(X_train, y_train, X_val, y_val, feature_names, seed):
    X_train_small, y_train_small = make_feature_rank_train_subset(
        X_train, y_train, FEATURE_RANK_MODEL_TRAIN_CAP, seed
    )
    X_val_small, y_val_small = make_feature_rank_eval_subset(
        X_val, y_val, FEATURE_RANK_PERM_CAP, seed + 1
    )

    baseline = LogisticRegression(
        solver="lbfgs",
        penalty="l2",
        C=1.0,
        max_iter=1500,
        random_state=seed,
    )
    baseline.fit(X_train_small, y_train_small)

    perm_results = permutation_importance(
        estimator=baseline,
        X=X_val_small,
        y=y_val_small,
        scoring=roc_auc_proba_scorer,
        n_repeats=FEATURE_RANK_PERM_REPEATS,
        random_state=seed,
        n_jobs=1,
        max_samples=1.0,
    )
    importance_vector = np.asarray(perm_results.importances_mean, dtype=float)
    return importance_vector



In [ ]:
# Convert importance scores into rank numbers, with rank 1 assigned to the largest importance.

def importance_to_rank_vector(importance_vector):
    order = np.argsort(-importance_vector)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(order) + 1)
    return ranks



In [ ]:
# Split outer calibration into final-fit, AUC-monitor, and threshold subsets, then cap the fit pool.

def split_final_train_monitor_threshold_indices(y_cal, monitor_size, threshold_size, train_cap, seed):
    y_cal = ensure_binary_int_labels(y_cal, "y_cal_final_split")
    n = len(y_cal)
    total_hold = min(monitor_size + threshold_size, max(2, n // 3))
    if total_hold >= n:
        raise ValueError("Final monitor/threshold sizes too large for outer calibration fold.")
    indices = np.arange(n)
    fit_pool_idx, internal_idx = train_test_split(
        indices,
        test_size=total_hold,
        stratify=y_cal,
        random_state=seed,
    )
    y_internal = y_cal[internal_idx]
    threshold_actual = min(threshold_size, max(1, len(internal_idx) // 2))
    monitor_idx, threshold_idx = train_test_split(
        internal_idx,
        test_size=threshold_actual,
        stratify=y_internal,
        random_state=seed + 1,
    )
    y_fit_pool = y_cal[fit_pool_idx]
    cap_idx_rel = make_cap_subset_indices(y_fit_pool, train_cap, seed + 2)
    fit_idx = fit_pool_idx[cap_idx_rel]
    return fit_idx, monitor_idx, threshold_idx



In [ ]:
# Train symbolic regression in generation chunks, save the best monitor-AUC model, and return run diagnostics.

def train_with_auc_monitoring(X_train, y_train, X_mon, y_mon, current_names, seed, outer_fold):
    model_params = BASE_FIXED_PARAMS.copy()
    model_params["feature_names"] = current_names
    model_params["random_state"] = seed

    model = RobustSymbolicClassifier(generations=GENERATION_STEP, **model_params)
    best_auc = -np.inf
    best_generation = 0
    tmp_path = os.path.join(OUTPUT_DIR, f"_tmp_best_sr_smote_outer{outer_fold}_{RUN_TAG}.pkl")
    no_improve_rounds = 0

    run_rows = []

    for current_gen in range(GENERATION_STEP, TOTAL_GENERATIONS + 1, GENERATION_STEP):
        model.generations = current_gen
        chunk_start = time.time()
        log(f"\n[Outer {outer_fold}] Fitting symbolic model up to generation {current_gen}/{TOTAL_GENERATIONS} ...")
        model.fit(X_train, y_train)
        chunk_min = (time.time() - chunk_start) / 60.0

        best_log_loss = np.nan
        avg_log_loss = np.nan
        avg_length = np.nan
        best_length = np.nan
        try:
            best_log_loss = model.run_details_["best_fitness"][-1]
            avg_log_loss = model.run_details_["average_fitness"][-1]
            avg_length = model.run_details_["average_length"][-1]
            best_length = model.run_details_["best_length"][-1]
        except Exception:
            pass

        try:
            y_mon_proba = model.predict_proba(X_mon)[:, 1]
            auc_monitor = safe_roc_auc(y_mon, y_mon_proba)
        except Exception as e:
            log(f"AUC monitoring prediction failed: {e}")
            auc_monitor = np.nan

        run_rows.append({
            "Outer_Fold": outer_fold,
            "Generation": current_gen,
            "Best_LogLoss": best_log_loss,
            "Average_LogLoss": avg_log_loss,
            "Best_Length": best_length,
            "Average_Length": avg_length,
            "Monitor_AUC": auc_monitor,
            "Chunk_Time_Min": chunk_min,
        })

        log(
            f"[Outer {outer_fold}] gen={current_gen} | "
            f"best_log_loss={best_log_loss:.6f} | avg_log_loss={avg_log_loss:.6f} | "
            f"monitor_auc={auc_monitor:.6f} | best_len={best_length} | "
            f"avg_len={avg_length:.2f} | chunk_time={chunk_min:.1f}m"
        )

        if np.isfinite(auc_monitor) and auc_monitor > (best_auc + MIN_AUC_IMPROVEMENT):
            best_auc = auc_monitor
            best_generation = current_gen
            no_improve_rounds = 0
            joblib.dump(model, tmp_path)
            log(f"[Outer {outer_fold}] New best monitor AUC: {best_auc:.6f} at generation {best_generation}")
        else:
            no_improve_rounds += 1
            log(f"[Outer {outer_fold}] No AUC improvement. patience={no_improve_rounds}/{EARLY_STOPPING_PATIENCE}")

        if no_improve_rounds >= EARLY_STOPPING_PATIENCE:
            log(f"[Outer {outer_fold}] Early stopping triggered at generation {current_gen}.")
            break

    if os.path.exists(tmp_path):
        best_model = joblib.load(tmp_path)
        try:
            os.remove(tmp_path)
        except OSError:
            pass
    else:
        best_model = model

    return best_model, best_generation, best_auc, pd.DataFrame(run_rows)



In [ ]:
# Summarize outer-fold metrics with mean, standard deviation, min, max, and missing counts.

def summarize_outer_results(df_outer):
    metric_cols = [
        "Accuracy", "Precision", "Recall", "Specificity", "NPV", "F1", "ROC_AUC",
        "Threshold", "Predicted_Positive_Rate", "Predicted_Negative_Rate",
        "Best_Generation", "Monitor_AUC", "Outer_SMOTE_Time_Min", "Outer_Time_Min",
    ]
    rows = []
    for col in metric_cols:
        if col in df_outer.columns:
            rows.append({
                "Metric": col,
                "Mean": df_outer[col].mean(skipna=True),
                "Std": df_outer[col].std(ddof=1, skipna=True),
                "Min": df_outer[col].min(skipna=True),
                "Max": df_outer[col].max(skipna=True),
                "Valid_Folds": int(df_outer[col].notna().sum()),
                "Missing_or_Undefined_Folds": int(df_outer[col].isna().sum()),
            })
    return pd.DataFrame(rows)



In [57]:

# ==============================================================================
# MAIN
# ==============================================================================


In [58]:

log(
    f"--- CONFIGURATION ---\n"
    f"RUN_TAG: {RUN_TAG}\n"
    f"RAW_DATA_PATH: {RAW_DATA_PATH}\n"
    f"TARGET_COL: {TARGET_COL}\n"
    f"N_OUTER_FOLDS: {N_OUTER_FOLDS}\n"
    f"N_INNER_FOLDS: {N_INNER_FOLDS}\n"
    f"TOP_K_FEATURES: {TOP_K_FEATURES}\n"
    f"OUTER_CALIBRATION_CAP: {OUTER_CALIBRATION_CAP}\n"
    f"INNER_TRAIN_CAP: {INNER_TRAIN_CAP}\n"
    f"FINAL_TRAIN_CAP: {FINAL_TRAIN_CAP}\n"
    f"FINAL_MONITOR_SIZE: {FINAL_MONITOR_SIZE}\n"
    f"FINAL_THRESHOLD_SIZE: {FINAL_THRESHOLD_SIZE}\n"
    f"SMOTE_SAMPLING_STRATEGY: {SMOTE_SAMPLING_STRATEGY}\n"
    f"SMOTE_K_NEIGHBORS: {SMOTE_K_NEIGHBORS}\n"
    f"SMOTE_IMPLEMENTATION: pure SMOTE after MinMax + one-hot preprocessing\n"
    f"NUMERIC_SCALER: MinMaxScaler\n"
    f"USE_SIGMOID_PROBA_CALIBRATION: {USE_SIGMOID_PROBA_CALIBRATION}\n"
    f"MIN_THRESHOLD: {MIN_THRESHOLD}\n"
    f"MIN_SPECIFICITY_FOR_THRESHOLD: {MIN_SPECIFICITY_FOR_THRESHOLD}\n"
    f"MAX_PREDICTED_POSITIVE_RATE: {MAX_PREDICTED_POSITIVE_RATE}\n"
    f"MIN_PREDICTED_NEGATIVE_RATE: {MIN_PREDICTED_NEGATIVE_RATE}\n"
    f"FEATURE_RANK_MODEL_TRAIN_CAP: {FEATURE_RANK_MODEL_TRAIN_CAP}\n"
    f"FEATURE_RANK_PERM_CAP: {FEATURE_RANK_PERM_CAP}\n"
    f"FEATURE_RANK_PERM_REPEATS: {FEATURE_RANK_PERM_REPEATS}\n"
    f"TOTAL_GENERATIONS: {TOTAL_GENERATIONS}\n"
    f"GENERATION_STEP: {GENERATION_STEP}\n"
    f"EARLY_STOPPING_PATIENCE: {EARLY_STOPPING_PATIENCE}\n"
    f"THRESHOLD_OBJECTIVE: {THRESHOLD_OBJECTIVE}\n"
    f"OUTPUT_DIR: {OUTPUT_DIR}\n"
    f"---------------------"
)


--- CONFIGURATION ---
RUN_TAG: sr_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained
RAW_DATA_PATH: ../data/processed/dados_uteis_refreshed_v3_memorysafe.pkl
TARGET_COL: ausencia
N_OUTER_FOLDS: 10
N_INNER_FOLDS: 10
TOP_K_FEATURES: 20
OUTER_CALIBRATION_CAP: 300000
INNER_TRAIN_CAP: 80000
FINAL_TRAIN_CAP: 150000
FINAL_MONITOR_SIZE: 30000
FINAL_THRESHOLD_SIZE: 30000
SMOTE_SAMPLING_STRATEGY: 0.5
SMOTE_K_NEIGHBORS: 3
SMOTE_IMPLEMENTATION: pure SMOTE after MinMax + one-hot preprocessing
NUMERIC_SCALER: MinMaxScaler
USE_SIGMOID_PROBA_CALIBRATION: True
MIN_THRESHOLD: 0.1
MIN_SPECIFICITY_FOR_THRESHOLD: 0.2
MAX_PREDICTED_POSITIVE_RATE: 0.7
MIN_PREDICTED_NEGATIVE_RATE: 0.05
FEATURE_RANK_MODEL_TRAIN_CAP: 80000
FEATURE_RANK_PERM_CAP: 20000
FEATURE_RANK_PERM_REPEATS: 3
TOTAL_GENERATIONS: 35
GENERATION_STEP: 5
EARLY_STOPPING_PATIENCE: 4
THRESHOLD_OBJECTIVE: youden_j
OUTPUT_DIR: ../data/processed
---------------------


In [59]:

log("Loading raw dataset...")
df = pd.read_pickle(RAW_DATA_PATH)
if not isinstance(df, pd.DataFrame):
    raise TypeError(f"Loaded object is {type(df).__name__}, not DataFrame")
if TARGET_COL not in df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not found")

log(f"Raw DataFrame shape: {df.shape}")
y_full = df[TARGET_COL].copy()
feature_drop_cols = [c for c in DROP_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in feature_drop_cols + [TARGET_COL]]
log(f"Number of feature columns before preprocessing: {len(feature_cols)}")
print_class_distribution("FULL DATASET DISTRIBUTION", y_full)


Loading raw dataset...
Raw DataFrame shape: (7113798, 42)
Number of feature columns before preprocessing: 25

FULL DATASET DISTRIBUTION
-------------------------
Total: 7.113.798
Class 0: 6.069.245 (85.3165%)
Class 1: 1.044.553 (14.6835%)
Imbalance ratio (maj/min): 5.8104:1


In [60]:

outer_skf = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=SEED)

outer_results = []
outer_feature_rows = []
inner_feature_rows = []
run_detail_rows = []
global_outer_selected_counter = Counter()

best_outer_fold_auc = -np.inf
best_outer_artifacts = None

for outer_fold, (cal_idx_full, val_idx) in enumerate(
    outer_skf.split(np.zeros(len(y_full)), ensure_binary_int_labels(y_full, "y_full_for_outer")),
    start=1,
):
    outer_start = time.time()
    log("\n" + "#" * 90)
    log(f"STARTING OUTER FOLD {outer_fold}/{N_OUTER_FOLDS}")
    log("#" * 90)

    y_cal_full = ensure_binary_int_labels(df.iloc[cal_idx_full][TARGET_COL], f"y_cal_full_outer_{outer_fold}")
    idx_cap_outer = make_cap_subset_indices(y_cal_full, OUTER_CALIBRATION_CAP, SEED + outer_fold)
    cal_idx = cal_idx_full[idx_cap_outer]

    X_cal_raw = df.iloc[cal_idx][feature_cols].copy()
    y_cal = ensure_binary_int_labels(df.iloc[cal_idx][TARGET_COL], f"y_cal_outer_{outer_fold}")
    X_val_raw = df.iloc[val_idx][feature_cols].copy()
    y_val = ensure_binary_int_labels(df.iloc[val_idx][TARGET_COL], f"y_val_outer_{outer_fold}")

    print_class_distribution(f"OUTER FOLD {outer_fold} - CALIBRATION", y_cal)
    print_class_distribution(f"OUTER FOLD {outer_fold} - VALIDATION", y_val)

    num_feats, cat_feats, cat_categories = build_outer_schema(X_cal_raw)
    category_maps, inverse_maps = learn_category_maps_from_schema(cat_feats, cat_categories)
    log(f"Outer fold {outer_fold} schema | numeric={len(num_feats)} | categorical={len(cat_feats)}")

    inner_skf = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=SEED + outer_fold)
    inner_importance_vectors = []
    inner_rank_vectors = []
    selected_counter_inner = Counter()
    inner_smote_times = []

    for inner_fold, (in_train_idx_full, in_test_idx) in enumerate(
        inner_skf.split(np.zeros(len(y_cal)), y_cal),
        start=1,
    ):
        log("\n" + "=" * 80)
        log(f"OUTER {outer_fold} | INNER {inner_fold}/{N_INNER_FOLDS}")
        log("=" * 80)

        y_in_train_full = ensure_binary_int_labels(y_cal[in_train_idx_full], f"y_in_train_full_{outer_fold}_{inner_fold}")
        idx_cap_inner = make_cap_subset_indices(y_in_train_full, INNER_TRAIN_CAP, SEED + outer_fold * 100 + inner_fold)
        in_train_idx = in_train_idx_full[idx_cap_inner]

        X_in_train_raw = X_cal_raw.iloc[in_train_idx].copy()
        y_in_train = ensure_binary_int_labels(y_cal[in_train_idx], f"y_in_train_{outer_fold}_{inner_fold}")
        X_in_test_raw = X_cal_raw.iloc[in_test_idx].copy()
        y_in_test = ensure_binary_int_labels(y_cal[in_test_idx], f"y_in_test_{outer_fold}_{inner_fold}")

        X_in_train_clean = clean_raw_like_dataframe(X_in_train_raw, num_feats, cat_feats, cat_categories)
        X_in_test_clean = clean_raw_like_dataframe(X_in_test_raw, num_feats, cat_feats, cat_categories)

        preprocessor_inner = build_preprocessor_from_schema(num_feats, cat_feats, cat_categories)
        preprocessor_inner.fit(X_in_train_clean)
        feature_names_inner = preprocessor_inner.get_feature_names_out().tolist()

        X_in_train = to_float32_array(preprocessor_inner.transform(X_in_train_clean))
        X_in_test = to_float32_array(preprocessor_inner.transform(X_in_test_clean))

        X_in_train_smote, y_in_train_smote, _, smote_time_min = apply_pure_smote_array(
            X_train=X_in_train,
            y_train=y_in_train,
            seed=SEED + outer_fold * 1000 + inner_fold,
            label=f"INNER PURE SMOTE outer={outer_fold} inner={inner_fold}",
        )
        inner_smote_times.append(smote_time_min)

        importance_vector = compute_lr_permutation_importance_vector(
            X_train=X_in_train_smote,
            y_train=y_in_train_smote,
            X_val=X_in_test,
            y_val=y_in_test,
            feature_names=feature_names_inner,
            seed=SEED + outer_fold * 10_000 + inner_fold,
        )
        rank_vector = importance_to_rank_vector(importance_vector)
        inner_importance_vectors.append(importance_vector)
        inner_rank_vectors.append(rank_vector)

        top_idx_inner = np.argsort(-importance_vector)[:TOP_K_FEATURES]
        top_names_inner = [feature_names_inner[i] for i in top_idx_inner]
        selected_counter_inner.update(top_names_inner)

        for rank_pos, idx_feat in enumerate(top_idx_inner, start=1):
            inner_feature_rows.append({
                "Outer_Fold": outer_fold,
                "Inner_Fold": inner_fold,
                "Rank_Within_Inner": rank_pos,
                "Feature_Index": int(idx_feat),
                "Feature": feature_names_inner[idx_feat],
                "Importance_Mean": float(importance_vector[idx_feat]),
                "SMOTE_Time_Min": float(smote_time_min),
            })

        del (
            X_in_train_raw,
            X_in_test_raw,
            X_in_train_clean,
            X_in_test_clean,
            X_in_train,
            X_in_train_smote,
            X_in_test,
        )
        gc.collect()

    # ------------------ Consolidate best feature subset ------------------
    importance_matrix = np.vstack(inner_importance_vectors)
    rank_matrix = np.vstack(inner_rank_vectors)
    mean_importance = importance_matrix.mean(axis=0)
    std_importance = importance_matrix.std(axis=0, ddof=1) if importance_matrix.shape[0] > 1 else np.zeros_like(mean_importance)
    mean_rank = rank_matrix.mean(axis=0)

    feature_names_outer = feature_names_inner
    inner_feature_summary = pd.DataFrame({
        "Feature_Index": np.arange(len(feature_names_outer)),
        "Feature": feature_names_outer,
        "Mean_Inner_Importance": mean_importance,
        "Std_Inner_Importance": std_importance,
        "Mean_Inner_Rank": mean_rank,
        "Selection_Frequency_Inner": [selected_counter_inner.get(f, 0) for f in feature_names_outer],
    }).sort_values(
        by=["Selection_Frequency_Inner", "Mean_Inner_Importance", "Mean_Inner_Rank"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    consolidated_idx = inner_feature_summary.head(TOP_K_FEATURES)["Feature_Index"].to_numpy(dtype=int)
    consolidated_features = inner_feature_summary.head(TOP_K_FEATURES)["Feature"].tolist()

    log(f"Outer fold {outer_fold} consolidated top-{TOP_K_FEATURES} features:")
    for i, f in enumerate(consolidated_features, start=1):
        log(f"  {i:02d}. {f}")

    for rank_pos, row in inner_feature_summary.head(TOP_K_FEATURES).reset_index(drop=True).iterrows():
        outer_feature_rows.append({
            "Outer_Fold": outer_fold,
            "Rank_Within_Outer": rank_pos + 1,
            "Feature_Index": int(row["Feature_Index"]),
            "Feature": row["Feature"],
            "Mean_Inner_Importance": float(row["Mean_Inner_Importance"]),
            "Std_Inner_Importance": float(row["Std_Inner_Importance"]),
            "Mean_Inner_Rank": float(row["Mean_Inner_Rank"]),
            "Selection_Frequency_Inner": int(row["Selection_Frequency_Inner"]),
        })
        global_outer_selected_counter[row["Feature"]] += 1

    # ------------------ Final outer training with train-side monitor/threshold ------------------
    fit_idx_rel, monitor_idx_rel, threshold_idx_rel = split_final_train_monitor_threshold_indices(
        y_cal=y_cal,
        monitor_size=FINAL_MONITOR_SIZE,
        threshold_size=FINAL_THRESHOLD_SIZE,
        train_cap=FINAL_TRAIN_CAP,
        seed=SEED + outer_fold * 50_000,
    )

    X_fit_raw = X_cal_raw.iloc[fit_idx_rel].copy()
    y_fit = ensure_binary_int_labels(y_cal[fit_idx_rel], f"y_fit_outer_{outer_fold}")
    X_monitor_raw = X_cal_raw.iloc[monitor_idx_rel].copy()
    y_monitor = ensure_binary_int_labels(y_cal[monitor_idx_rel], f"y_monitor_outer_{outer_fold}")
    X_threshold_raw = X_cal_raw.iloc[threshold_idx_rel].copy()
    y_threshold = ensure_binary_int_labels(y_cal[threshold_idx_rel], f"y_threshold_outer_{outer_fold}")

    X_fit_clean = clean_raw_like_dataframe(X_fit_raw, num_feats, cat_feats, cat_categories)
    X_monitor_clean = clean_raw_like_dataframe(X_monitor_raw, num_feats, cat_feats, cat_categories)
    X_threshold_clean = clean_raw_like_dataframe(X_threshold_raw, num_feats, cat_feats, cat_categories)
    X_val_clean = clean_raw_like_dataframe(X_val_raw, num_feats, cat_feats, cat_categories)

    preprocessor_outer = build_preprocessor_from_schema(num_feats, cat_feats, cat_categories)
    preprocessor_outer.fit(X_fit_clean)
    feature_names_outer_final = preprocessor_outer.get_feature_names_out().tolist()

    if feature_names_outer_final != feature_names_outer:
        raise RuntimeError(f"Feature schema mismatch in outer fold {outer_fold}.")

    X_fit = to_float32_array(preprocessor_outer.transform(X_fit_clean))
    X_monitor = to_float32_array(preprocessor_outer.transform(X_monitor_clean))
    X_threshold = to_float32_array(preprocessor_outer.transform(X_threshold_clean))
    X_val = to_float32_array(preprocessor_outer.transform(X_val_clean))

    X_fit_smote, y_fit_smote, _, outer_smote_time_min = apply_pure_smote_array(
        X_train=X_fit,
        y_train=y_fit,
        seed=SEED + outer_fold * 1_000_000,
        label=f"OUTER PURE SMOTE outer={outer_fold}",
    )

    X_fit_sel = X_fit_smote[:, consolidated_idx]
    X_monitor_sel = X_monitor[:, consolidated_idx]
    X_threshold_sel = X_threshold[:, consolidated_idx]
    X_val_sel = X_val[:, consolidated_idx]

    best_model, best_generation, best_auc_monitor, df_run_details_outer = train_with_auc_monitoring(
        X_train=X_fit_sel,
        y_train=y_fit_smote,
        X_mon=X_monitor_sel,
        y_mon=y_monitor,
        current_names=consolidated_features,
        seed=SEED + outer_fold * 2_000_000,
        outer_fold=outer_fold,
    )
    run_detail_rows.append(df_run_details_outer)

    y_monitor_proba_raw = best_model.predict_proba(X_monitor_sel)[:, 1]
    y_threshold_proba_raw = best_model.predict_proba(X_threshold_sel)[:, 1]
    y_val_proba_raw = best_model.predict_proba(X_val_sel)[:, 1]

    calibrator = None
    calibration_method = "none_raw_symbolic_proba"
    if USE_SIGMOID_PROBA_CALIBRATION:
        calibrator = fit_sigmoid_calibrator(
            y_calib=y_monitor,
            proba_calib=y_monitor_proba_raw,
            seed=SEED + outer_fold * 3_000_000,
        )
        if calibrator is not None:
            calibration_method = "sigmoid_platt_on_monitor"

    y_threshold_proba = apply_sigmoid_calibrator(calibrator, y_threshold_proba_raw)
    y_val_proba = apply_sigmoid_calibrator(calibrator, y_val_proba_raw)

    best_threshold_row, df_thresholds = optimize_threshold_constrained(
        y_true=y_threshold,
        y_proba=y_threshold_proba,
        objective=THRESHOLD_OBJECTIVE,
    )
    selected_threshold = float(best_threshold_row["Threshold"])

    df_thresholds.insert(0, "Outer_Fold", outer_fold)
    threshold_path = os.path.join(OUTPUT_DIR, f"threshold_search_outer{outer_fold}_{RUN_TAG}.csv")
    df_thresholds.to_csv(threshold_path, index=False)

    y_val_pred = (y_val_proba >= selected_threshold).astype(np.uint8)
    metrics_outer = calculate_metrics(y_val, y_val_pred, y_val_proba)

    log(
        f"OUTER FOLD {outer_fold} CONFUSION MATRIX [[TN, FP], [FN, TP]] = "
        f"[[{metrics_outer['TN']}, {metrics_outer['FP']}], [{metrics_outer['FN']}, {metrics_outer['TP']}]]"
    )

    outer_elapsed_min = (time.time() - outer_start) / 60.0

    equation = str(best_model._program)
    outer_row = {
        "Outer_Fold": outer_fold,
        "Threshold": selected_threshold,
        "Threshold_Objective": THRESHOLD_OBJECTIVE,
        "ThresholdTune_F1": float(best_threshold_row["F1"]),
        "ThresholdTune_Recall": float(best_threshold_row["Recall"]),
        "ThresholdTune_Precision": float(best_threshold_row["Precision"]),
        "ThresholdTune_Specificity": float(best_threshold_row["Specificity"]),
        "ThresholdTune_Youden_J": float(best_threshold_row["Youden_J"]),
        "ThresholdTune_NPV": float(best_threshold_row.get("NPV", np.nan)) if not pd.isna(best_threshold_row.get("NPV", np.nan)) else np.nan,
        "ThresholdTune_Balanced_Accuracy": float(best_threshold_row.get("Balanced_Accuracy", np.nan)),
        "ThresholdTune_Predicted_Positive_Rate": float(best_threshold_row.get("Predicted_Positive_Rate", np.nan)),
        "ThresholdTune_Predicted_Negative_Rate": float(best_threshold_row.get("Predicted_Negative_Rate", np.nan)),
        "Threshold_Selection_Mode": best_threshold_row.get("Threshold_Selection_Mode", "unknown"),
        "Threshold_Valid_Candidates": int(best_threshold_row.get("Threshold_Valid_Candidates", -1)),
        "Threshold_Total_Candidates": int(best_threshold_row.get("Threshold_Total_Candidates", -1)),
        "Calibration_Method": calibration_method,
        "Probability_Calibrated": bool(calibrator is not None),
        "Best_Generation": int(best_generation),
        "Monitor_AUC": float(best_auc_monitor),
        "Outer_SMOTE_Time_Min": float(outer_smote_time_min),
        "Inner_SMOTE_Time_Min_Mean": float(np.mean(inner_smote_times)),
        "Outer_Time_Min": float(outer_elapsed_min),
        "N_Selected_Features": TOP_K_FEATURES,
        "Selected_Features": " | ".join(consolidated_features),
        "Equation": equation,
        "Model": "SymbolicRegression",
        "Technique": "Raw_10FoldCV + PureSMOTE_PostMinMaxOHE + NestedTop20Permutation",
        "Mode": RUN_TAG,
        **metrics_outer,
    }
    outer_results.append(outer_row)

    log(
        f"OUTER FOLD {outer_fold} COMPLETE | "
        f"F1={metrics_outer['F1']:.6f} | Recall={metrics_outer['Recall']:.6f} | "
        f"Precision={metrics_outer['Precision']:.6f} | Specificity={metrics_outer['Specificity']:.6f} | "
        f"NPV={metrics_outer['NPV'] if not pd.isna(metrics_outer['NPV']) else np.nan:.6f} | "
        f"ROC_AUC={metrics_outer['ROC_AUC']:.6f} | Threshold={selected_threshold:.6f} | "
        f"PredPosRate={metrics_outer['Predicted_Positive_Rate']:.4f} | "
        f"PredNegRate={metrics_outer['Predicted_Negative_Rate']:.4f} | "
        f"ThresholdMode={best_threshold_row['Threshold_Selection_Mode']} | "
        f"Calibration={calibration_method} | BestGen={best_generation} | Time={outer_elapsed_min:.1f}m"
    )

    if metrics_outer["ROC_AUC"] > best_outer_fold_auc:
        best_outer_fold_auc = metrics_outer["ROC_AUC"]
        best_outer_artifacts = {
            "Outer_Fold": outer_fold,
            "Model": best_model,
            "Preprocessor": preprocessor_outer,
            "Selected_Feature_Indices": consolidated_idx,
            "Selected_Feature_Names": consolidated_features,
            "X_val_sel": X_val_sel,
            "y_val": y_val,
            "Calibrator": calibrator,
            "Calibration_Method": calibration_method,
            "Selected_Threshold": selected_threshold,
            "Threshold_Objective": THRESHOLD_OBJECTIVE,
            "Threshold_Row": best_threshold_row,
            "Equation": equation,
        }

    del (
        X_cal_raw,
        X_val_raw,
        X_fit_raw,
        X_monitor_raw,
        X_threshold_raw,
        X_fit_clean,
        X_monitor_clean,
        X_threshold_clean,
        X_val_clean,
        X_fit,
        X_fit_smote,
        X_monitor,
        X_threshold,
        X_val,
        X_fit_sel,
        X_monitor_sel,
        X_threshold_sel,
        X_val_sel,
        best_model,
        calibrator,
        y_monitor_proba_raw,
        y_threshold_proba_raw,
        y_val_proba_raw,
        y_threshold_proba,
        y_val_proba,
        y_val_pred,
    )
    gc.collect()




##########################################################################################
STARTING OUTER FOLD 1/10
##########################################################################################

OUTER FOLD 1 - CALIBRATION
--------------------------
Total: 300.000
Class 0: 255.950 (85.3167%)
Class 1: 44.050 (14.6833%)
Imbalance ratio (maj/min): 5.8104:1

OUTER FOLD 1 - VALIDATION
-------------------------
Total: 711.380
Class 0: 606.925 (85.3166%)
Class 1: 104.455 (14.6834%)
Imbalance ratio (maj/min): 5.8104:1
Outer fold 1 schema | numeric=15 | categorical=10

OUTER 1 | INNER 1/10
INNER PURE SMOTE outer=1 inner=1 finished in 0.01m | before class0=68253, class1=11747 | after class0=68253, class1=34126

OUTER 1 | INNER 2/10
INNER PURE SMOTE outer=1 inner=2 finished in 0.00m | before class0=68253, class1=11747 | after class0=68253, class1=34126

OUTER 1 | INNER 3/10
INNER PURE SMOTE outer=1 inner=3 finished in 0.01m | before class0=68253, class1=11747 | after class0=68253, cl

In [61]:

# ==============================================================================
# SAVE RESULTS
# ==============================================================================


In [62]:

df_outer = pd.DataFrame(outer_results)
df_outer_summary = summarize_outer_results(df_outer)
df_confusion_matrices = df_outer[[
    "Outer_Fold", "TN", "FP", "FN", "TP",
    "Real_Negative", "Real_Positive", "Predicted_Negative", "Predicted_Positive",
    "Predicted_Negative_Rate", "Predicted_Positive_Rate",
    "Accuracy", "Precision", "Recall", "Specificity", "NPV", "F1", "ROC_AUC", "Threshold",
]].copy()
df_pooled_summary = summarize_pooled_confusion(df_outer)
df_outer_features = pd.DataFrame(outer_feature_rows)
df_inner_features = pd.DataFrame(inner_feature_rows)
df_run_details = pd.concat(run_detail_rows, ignore_index=True) if run_detail_rows else pd.DataFrame()

global_feature_summary_rows = []
for feature, count in global_outer_selected_counter.most_common():
    global_feature_summary_rows.append({
        "Feature": feature,
        "Outer_Fold_Frequency": count,
        "Outer_Fold_Frequency_Pct": count / N_OUTER_FOLDS * 100.0,
    })
df_global_feature_summary = pd.DataFrame(global_feature_summary_rows)
final_global_top20 = df_global_feature_summary.head(TOP_K_FEATURES).copy()
final_global_top20["Global_Rank"] = np.arange(1, len(final_global_top20) + 1)

outer_results_path = os.path.join(OUTPUT_DIR, f"outer_fold_results_{RUN_TAG}.csv")
outer_summary_path = os.path.join(OUTPUT_DIR, f"outer_fold_summary_{RUN_TAG}.csv")
confusion_matrices_path = os.path.join(OUTPUT_DIR, f"confusion_matrices_{RUN_TAG}.csv")
pooled_summary_path = os.path.join(OUTPUT_DIR, f"pooled_confusion_summary_{RUN_TAG}.csv")
outer_features_path = os.path.join(OUTPUT_DIR, f"outer_fold_selected_features_{RUN_TAG}.csv")
inner_features_path = os.path.join(OUTPUT_DIR, f"inner_selected_features_{RUN_TAG}.csv")
run_details_path = os.path.join(OUTPUT_DIR, f"symbolic_run_details_{RUN_TAG}.csv")
global_feature_summary_path = os.path.join(OUTPUT_DIR, f"global_feature_summary_{RUN_TAG}.csv")
final_global_top20_path = os.path.join(OUTPUT_DIR, f"final_global_top20_{RUN_TAG}.csv")
config_path = os.path.join(OUTPUT_DIR, f"config_summary_{RUN_TAG}.json")
best_outer_model_path = os.path.join(OUTPUT_DIR, f"best_outer_model_{RUN_TAG}.pkl")
best_outer_artifacts_path = os.path.join(OUTPUT_DIR, f"best_outer_artifacts_{RUN_TAG}.pkl")

df_outer.to_csv(outer_results_path, index=False)
df_outer_summary.to_csv(outer_summary_path, index=False)
df_confusion_matrices.to_csv(confusion_matrices_path, index=False)
df_pooled_summary.to_csv(pooled_summary_path, index=False)
df_outer_features.to_csv(outer_features_path, index=False)
df_inner_features.to_csv(inner_features_path, index=False)
df_run_details.to_csv(run_details_path, index=False)
df_global_feature_summary.to_csv(global_feature_summary_path, index=False)
final_global_top20.to_csv(final_global_top20_path, index=False)


In [63]:

config_summary = {
    "SEED": SEED,
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "TARGET_COL": TARGET_COL,
    "RUN_TAG": RUN_TAG,
    "DROP_COLS": DROP_COLS,
    "KNOWN_CATEGORICAL_COLS": KNOWN_CATEGORICAL_COLS,
    "N_OUTER_FOLDS": N_OUTER_FOLDS,
    "N_INNER_FOLDS": N_INNER_FOLDS,
    "TOP_K_FEATURES": TOP_K_FEATURES,
    "OUTER_CALIBRATION_CAP": OUTER_CALIBRATION_CAP,
    "INNER_TRAIN_CAP": INNER_TRAIN_CAP,
    "FINAL_MONITOR_SIZE": FINAL_MONITOR_SIZE,
    "FINAL_THRESHOLD_SIZE": FINAL_THRESHOLD_SIZE,
    "FINAL_TRAIN_CAP": FINAL_TRAIN_CAP,
    "SMOTE_SAMPLING_STRATEGY": SMOTE_SAMPLING_STRATEGY,
    "SMOTE_K_NEIGHBORS": SMOTE_K_NEIGHBORS,
    "SMOTE_IMPLEMENTATION": "Pure SMOTE after MinMaxScaler + OneHotEncoder",
    "NUMERIC_SCALER": "MinMaxScaler",
    "USE_SIGMOID_PROBA_CALIBRATION": USE_SIGMOID_PROBA_CALIBRATION,
    "MIN_THRESHOLD": MIN_THRESHOLD,
    "MIN_SPECIFICITY_FOR_THRESHOLD": MIN_SPECIFICITY_FOR_THRESHOLD,
    "MIN_PRECISION_MARGIN_OVER_PREVALENCE": MIN_PRECISION_MARGIN_OVER_PREVALENCE,
    "MAX_PREDICTED_POSITIVE_RATE": MAX_PREDICTED_POSITIVE_RATE,
    "MIN_PREDICTED_NEGATIVE_RATE": MIN_PREDICTED_NEGATIVE_RATE,
    "FEATURE_RANK_MODEL_TRAIN_CAP": FEATURE_RANK_MODEL_TRAIN_CAP,
    "FEATURE_RANK_PERM_CAP": FEATURE_RANK_PERM_CAP,
    "FEATURE_RANK_PERM_REPEATS": FEATURE_RANK_PERM_REPEATS,
    "TOTAL_GENERATIONS": TOTAL_GENERATIONS,
    "GENERATION_STEP": GENERATION_STEP,
    "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
    "MIN_AUC_IMPROVEMENT": MIN_AUC_IMPROVEMENT,
    "BASE_FIXED_PARAMS": BASE_FIXED_PARAMS,
    "THRESHOLD_OBJECTIVE": THRESHOLD_OBJECTIVE,
    "MANUAL_THRESHOLDS": MANUAL_THRESHOLDS,
}
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_summary, f, ensure_ascii=False, indent=2, default=str)

if best_outer_artifacts is not None:
    joblib.dump(best_outer_artifacts["Model"], best_outer_model_path)
    joblib.dump(best_outer_artifacts, best_outer_artifacts_path)

if RUN_OUTER_BEST_FOLD_PERMUTATION and best_outer_artifacts is not None:
    X_perm, y_perm = make_feature_rank_eval_subset(
        best_outer_artifacts["X_val_sel"],
        best_outer_artifacts["y_val"],
        OUTER_PERM_MAX_SAMPLES,
        SEED + 999,
    )
    perm_results = permutation_importance(
        estimator=best_outer_artifacts["Model"],
        X=X_perm,
        y=y_perm,
        scoring=roc_auc_proba_scorer,
        n_repeats=OUTER_PERM_REPEATS,
        random_state=SEED,
        n_jobs=1,
        max_samples=1.0,
    )
    df_best_perm = pd.DataFrame({
        "Feature": best_outer_artifacts["Selected_Feature_Names"],
        "Importance_Mean": perm_results.importances_mean,
        "Importance_Std": perm_results.importances_std,
    }).sort_values(by="Importance_Mean", ascending=False)
    best_perm_path = os.path.join(OUTPUT_DIR, f"best_outer_perm_importance_{RUN_TAG}.csv")
    df_best_perm.to_csv(best_perm_path, index=False)


In [64]:

log("\n" + "=" * 90)
log("Pure SMOTE + Symbolic Regression nested CV workflow complete.")
log("=" * 90)
log(f"Outer fold results:         {outer_results_path}")
log(f"Outer fold summary:         {outer_summary_path}")
log(f"Confusion matrices:         {confusion_matrices_path}")
log(f"Pooled confusion summary:   {pooled_summary_path}")
log(f"Outer selected features:    {outer_features_path}")
log(f"Inner selected features:    {inner_features_path}")
log(f"Symbolic run details:       {run_details_path}")
log(f"Global feature summary:     {global_feature_summary_path}")
log(f"Final global top-{TOP_K_FEATURES}:       {final_global_top20_path}")
log(f"Config summary:             {config_path}")
if best_outer_artifacts is not None:
    log(f"Best outer model:           {best_outer_model_path}")
    log(f"Best outer artifacts:       {best_outer_artifacts_path}")
    log(f"Best outer fold:            {best_outer_artifacts['Outer_Fold']}")
    log(f"Best outer threshold:       {best_outer_artifacts['Selected_Threshold']:.6f}")
    log(f"Best outer equation:        {best_outer_artifacts['Equation']}")



Pure SMOTE + Symbolic Regression nested CV workflow complete.
Outer fold results:         ../data/processed/outer_fold_results_sr_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Outer fold summary:         ../data/processed/outer_fold_summary_sr_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Confusion matrices:         ../data/processed/confusion_matrices_sr_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Pooled confusion summary:   ../data/processed/pooled_confusion_summary_sr_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Outer selected features:    ../data/processed/outer_fold_selected_features_sr_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Inner selected features:    ../data/processed/inner_selected_features_sr_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Symbolic run details:       ../data/processed/symbolic_run_details_sr_smote_refres

In [65]:

log("\nOUTER SUMMARY")
log(str(df_outer_summary))



OUTER SUMMARY
                     Metric      Mean       Std       Min        Max  \
0                  Accuracy  0.716159  0.119956  0.514971   0.853164   
1                 Precision  0.200511  0.074516  0.000000   0.257269   
2                    Recall  0.332656  0.236946  0.000000   0.689397   
3               Specificity  0.782162  0.180669  0.497487   1.000000   
4                       NPV  0.875541  0.017730  0.853164   0.908291   
5                        F1  0.225671  0.089460  0.000000   0.313812   
6                   ROC_AUC  0.569719  0.045916  0.500000   0.647339   
7                 Threshold  0.148164  0.032678  0.130000   0.238864   
8   Predicted_Positive_Rate  0.234697  0.188547  0.000000   0.519260   
9   Predicted_Negative_Rate  0.765303  0.188547  0.480740   1.000000   
10          Best_Generation  7.500000  3.535534  5.000000  15.000000   
11              Monitor_AUC  0.569404  0.045318  0.500000   0.647653   
12     Outer_SMOTE_Time_Min  0.016549  0.000136  

In [66]:

log("\nCONFUSION MATRICES")
log(str(df_confusion_matrices))



CONFUSION MATRICES
   Outer_Fold      TN      FP      FN     TP  Real_Negative  Real_Positive  \
0           1  515823   91102   76725  27730         606925         104455   
1           2  544440   62485   84737  19718         606925         104455   
2           3  355408  251517   38206  66249         606925         104455   
3           4  301937  304988   40052  64403         606925         104455   
4           5  544111   62814   84801  19654         606925         104455   
5           6  606924       0  104456      0         606924         104456   
6           7  456908  150016   64274  40182         606924         104456   
7           8  544602   62322   84686  19770         606924         104456   
8           9  555654   51270   86696  17759         606924         104455   
9          10  321328  285596   32444  72011         606924         104455   

   Predicted_Negative  Predicted_Positive  Predicted_Negative_Rate  \
0              592548              118832          

In [67]:

log("\nPOOLED CONFUSION SUMMARY")
log(str(df_pooled_summary.T))


POOLED CONFUSION SUMMARY
                                    0
TP                       3.474760e+05
TN                       4.747135e+06
FP                       1.322110e+06
FN                       6.970770e+05
Accuracy                 7.161591e-01
Precision                2.081211e-01
Recall                   3.326552e-01
Specificity              7.821624e-01
NPV                      8.719600e-01
F1                       2.560488e-01
Balanced_Accuracy        5.574088e-01
Predicted_Positive       1.669586e+06
Predicted_Negative       5.444212e+06
Real_Positive            1.044553e+06
Real_Negative            6.069245e+06
Predicted_Positive_Rate  2.346969e-01
Predicted_Negative_Rate  7.653031e-01
